In [2]:
import time
import re
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def clean_price(price_str):
    if not price_str: return 0
    nums = "".join(re.findall(r'\d+', str(price_str)))
    return int(nums) if nums else 0

def crawl_dalock_exact_list():
    print("🚀 다락 지점 목록 웹 파싱을 시작합니다...")
    
    chrome_options = Options()
    chrome_options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    
    driver = webdriver.Chrome(options=chrome_options)
    
    # 1. 대상 URL 접속
    target_url = "https://www.dalock.kr/service/location?productTypes=COMPACT%2CCUBE%2CSLIM%2CSMALL%2CMINI%2CMEDIUM%2CLARGE"
    driver.get(target_url)
    
    # 2. 페이지 및 지도/목록 요소가 완전히 로드될 때까지 최대 10초 대기
    print("⏳ 브라우저가 콘텐츠를 로딩 중입니다...")
    try:
        # 화면에 지점 리스트 영역이나 특정 텍스트가 나타날 때까지 대기합니다.
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
    except Exception:
        print("[경고] 로딩 대기 시간 초과. 계속 진행합니다.")
    
    time.sleep(4) # 비동기 데이터 추가 로드를 위해 안정적으로 4초 더 대기
    
    # 3. 스크롤을 여러 번 내려 리스트 바닥까지 전체 215개 지점 활성화
    print("📜 왼쪽 지점 목록 리스트를 확보하기 위해 페이지 다운 수행 중...")
    # 브라우저 창 크기를 키워 더 많은 리스트가 한 번에 보이도록 설정
    driver.set_window_size(1400, 1000)
    
    for scroll_idx in range(10):
        # 전체 화면 스크롤 및 목록 스크롤을 동시에 유도합니다.
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        try:
            # 다락의 왼쪽 리스트 컨테이너 혹은 스크롤 가능한 영역을 찾아 강제 스크롤
            list_container = driver.find_element(By.XPATH, "//*[contains(@class, 'list') or contains(@class, 'Container')]")
            driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight;", list_container)
        except:
            pass
        time.sleep(0.8)
        
    # 4. 전체 로드된 HTML 소스 덤프 및 BeautifulSoup 변환
    soup = BeautifulSoup(driver.page_source, "html.parser")
    driver.quit()
    
    all_data = []
    
    # 💡 [핵심] 텍스트 전체 분석 대신, '지점 카드' 역할을 하는 구역을 감지합니다.
    # 다락의 지점 리스트는 보통 특정 class명 묶음이나, <a> 태그 혹은 <div> 태그 단위로 반복됩니다.
    # 주소나 요금 정보가 함께 들어있는 '부모 구역'들을 모두 찾아냅니다.
    print("🔍 지점 카드 구역 수집 및 개별 매핑 중...")
    
    # 웹페이지 내에서 '원~' 혹은 '월'과 지점명이 함께 묶여 있는 모든 div 구역을 탐색
    card_elements = soup.find_all(lambda tag: tag.name == 'div' and tag.find(text=re.compile(r'점$')) and tag.find(text=re.compile(r'원~')))
    
    # 만약 위의 정밀 탐색으로 안 잡힐 경우, '원~' 텍스트가 있는 모든 엘리먼트 주변을 역추적합니다.
    if not card_elements:
        price_tags = soup.find_all(text=re.compile(r'원~'))
        card_elements = [p.parent.parent.parent for p in price_tags if p.parent]
        
    print(f"📊 탐지된 지점 데이터 구역 수: {len(card_elements)}개")
    
    for card in card_elements:
        card_text = card.get_text(separator="\n")
        lines = [line.strip() for line in card_text.split("\n") if line.strip()]
        
        # 한 카드 안에서 지점명과 가격 패턴을 안전하게 가출해 냅니다.
        branch_name = None
        
        # 1) 지점명 찾기 (예: '강남역점', '가든파이브점')
        for line in lines:
            if line.endswith("점") and not any(skip in line for skip in ["이용지점", "추천지점", "등록지점", "해당지점"]):
                branch_name = line
                break
                
        if not branch_name:
            continue
            
        # 2) 해당 지점 카드 내부의 유닛 및 가격 라인들 매칭
        for line in lines:
            if "원~" in line:
                # 패턴 예시: "큐브 월 77,200원~" 혹은 "슬림 54,000원~"
                unit_match = re.search(r'([가-힣a-zA-Z]+).+?([\d,]+)원~', line)
                if unit_match:
                    unit_type = unit_match.group(1).replace("월", "").strip()
                    price_val = clean_price(unit_match.group(2))
                    
                    all_data.append({
                        "branch_name": branch_name,
                        "unit_type": unit_type,
                        "discount_price": price_val
                    })
                else:
                    # 정규식에 안 걸리더라도 숫자와 텍스트가 있으면 분리 시도
                    price_val = clean_price(line)
                    # 가격 앞 단어를 유닛 타입으로 추정 (예: '큐브77,200원~' -> '큐브')
                    unit_type = line.split("원")[0]
                    unit_type = re.sub(r'[\d,\s월~]', '', unit_type)
                    
                    if price_val:
                        all_data.append({
                            "branch_name": branch_name,
                            "unit_type": unit_type if unit_type else "대표유닛",
                            "discount_price": price_val
                        })

    # 5. 결과 저장 및 파일 출력
    if all_data:
        df = pd.DataFrame(all_data)
        
        # 완벽한 데이터 정제 (중복 행 제거)
        df = df.drop_duplicates(subset=['branch_name', 'unit_type'], keep='first')
        df = df[df['branch_name'].str.endswith("점")]
        
        file_name = "dalock_list_cards_prices.csv"
        df.to_csv(file_name, index=False, encoding="utf-8-sig")
        print(f"\n🎉 [성공] 총 {df['branch_name'].nunique()}개 지점 요금 필터링 완료!")
        print(f"💾 엑셀 저장 파일명: {file_name}")
    else:
        print("\n😢 여전히 감지되지 않습니다. 다락 사이트의 보안 토큰이나 Next.js 내부 스크립트 구조(JSON 데이터) 영역을 직접 파싱하는 코드로 전환해야 할 수 있습니다.")

if __name__ == "__main__":
    crawl_dalock_exact_list()

🚀 다락 지점 목록 웹 파싱을 시작합니다...
⏳ 브라우저가 콘텐츠를 로딩 중입니다...
📜 왼쪽 지점 목록 리스트를 확보하기 위해 페이지 다운 수행 중...
🔍 지점 카드 구역 수집 및 개별 매핑 중...


C:\Users\SAMSUNG\AppData\Local\Temp\ipykernel_32704\2392329665.py:69: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  card_elements = soup.find_all(lambda tag: tag.name == 'div' and tag.find(text=re.compile(r'점$')) and tag.find(text=re.compile(r'원~')))


📊 탐지된 지점 데이터 구역 수: 820개

🎉 [성공] 총 204개 지점 요금 필터링 완료!
💾 엑셀 저장 파일명: dalock_list_cards_prices.csv
